# Web Search Tool with LangChain Agent

## Overview

This notebook demonstrates the same Web Search Tool integration using LangChain and LangGraph instead of Strands. It uses `langchain-mcp-adapters` to connect to the AgentCore Gateway and `create_react_agent` from LangGraph for the ReAct agent loop.

You will:
1. Connect to the Gateway using `MultiServerMCPClient` from `langchain-mcp-adapters`
2. Discover Web Search tools via MCP
3. Build a ReAct agent with `ChatBedrockConverse`
4. Run web-grounded Q&A

### Tutorial Details

| Information | Details |
|:------------|:--------|
| Tutorial type | Interactive (Jupyter Notebook) |
| AgentCore components | AgentCore Gateway |
| Agentic framework | LangChain + LangGraph |
| LLM model | Anthropic Claude Sonnet 4 (`us.anthropic.claude-sonnet-4-20250514-v1:0`) |
| Tutorial vertical | Cross-vertical |
| Example complexity | Easy |
| SDK used | boto3, langchain-aws, langchain-mcp-adapters |

### How It Works

```
User question
      │
      ▼
LangGraph ReAct Agent (ChatBedrockConverse)
      │  calls WebSearch tool
      ▼
MultiServerMCPClient  →  AgentCore Gateway  →  Web Search Backend
      │
      ▼
Structured results  →  Final answer with citations
```

## Prerequisites

Complete notebook `01-web-search-gateway-setup.ipynb` first and set the environment variables it prints.

### Required IAM permissions

~~~json
{
  "Effect": "Allow",
  "Action": "bedrock:InvokeModel",
  "Resource": "*"
}
~~~

## 1. Install Dependencies

In [ ]:
!pip install --upgrade -r requirements.txt --quiet

## 2. Configuration

In [ ]:
import asyncio
import os
import requests

REGION = os.environ.get("AWS_DEFAULT_REGION", "us-east-1")
GATEWAY_URL = os.environ["AGENTCORE_GATEWAY_URL"]
COGNITO_DOMAIN = os.environ["COGNITO_DOMAIN"]
CLIENT_ID = os.environ["COGNITO_CLIENT_ID"]
CLIENT_SECRET = os.environ["COGNITO_CLIENT_SECRET"]
SCOPE_STRING = os.environ["COGNITO_SCOPE"]
MODEL_ID = "us.anthropic.claude-sonnet-4-20250514-v1:0"

print(f"Gateway URL : {GATEWAY_URL}")
print(f"Region      : {REGION}")
print(f"Model       : {MODEL_ID}")

## 3. Token Helper

In [ ]:
def get_token():
    """Retrieve a fresh OAuth access token from Cognito."""
    url = f"https://{COGNITO_DOMAIN}.auth.{REGION}.amazoncognito.com/oauth2/token"
    resp = requests.post(
        url,
        headers={"Content-Type": "application/x-www-form-urlencoded"},
        data={
            "grant_type": "client_credentials",
            "client_id": CLIENT_ID,
            "client_secret": CLIENT_SECRET,
            "scope": SCOPE_STRING
        }
    )
    resp.raise_for_status()
    return resp.json()["access_token"]

token = get_token()
print("Token obtained ✓")

## 4. Create LangChain Agent with MCP Tools

`MultiServerMCPClient` connects to one or more MCP servers and converts their tools into LangChain-compatible tool objects. `create_react_agent` builds a LangGraph ReAct agent with those tools.

In [ ]:
from langchain_aws import ChatBedrockConverse
from langchain_mcp_adapters.client import MultiServerMCPClient
from langgraph.prebuilt import create_react_agent


async def run_agent(query: str) -> str:
    """Run the LangChain agent with a web search query."""
    model = ChatBedrockConverse(
        model=MODEL_ID,
        region_name=REGION,
        temperature=0.7,
        max_tokens=1024
    )

    async with MultiServerMCPClient({
        "web-search": {
            "transport": "streamable_http",
            "url": GATEWAY_URL,
            "headers": {"Authorization": f"Bearer {get_token()}"}
        }
    }) as client:
        tools = client.get_tools()
        print(f"Discovered {len(tools)} tool(s): {[t.name for t in tools]}")

        agent = create_react_agent(model, tools=tools)
        result = await agent.ainvoke({
            "messages": [{"role": "user", "content": query}]
        })

        final_message = result["messages"][-1]
        return final_message.content if hasattr(final_message, "content") else str(final_message)


print("Agent function defined ✓")

## 5. Ask a Real-Time Question

In [ ]:
QUERY = "What is today's news around the world?"

print(f"Query: {QUERY}")
print("-" * 60)

answer = await run_agent(QUERY)

print("\n" + "=" * 60)
print("LANGCHAIN AGENT RESPONSE")
print("=" * 60)
print(answer)

## 6. Try More Queries

In [ ]:
demo_queries = [
    "What are the latest AI model releases in 2026?",
    "What is the current EUR/USD exchange rate?",
    "What new features were announced in Python 3.14?"
]

for query in demo_queries:
    print(f"\n{'=' * 60}")
    print(f"Query: {query}")
    print("=" * 60)
    answer = await run_agent(query)
    print(answer)

## Conclusion

In this notebook you:
- Used `langchain-mcp-adapters` to connect to the AgentCore Gateway
- Built a LangGraph ReAct agent with `ChatBedrockConverse`
- Got grounded, cited answers using the same Gateway from notebook 01

The Gateway is framework-agnostic — Strands, LangChain, or any MCP client works the same way.

**Next**: Explore the advanced examples in `04-advanced-examples/` for multi-turn search patterns.